# Agentic Rag Module

In [2]:
# Importing libraries and create openai client
import os
from dotenv import load_dotenv
load_dotenv()
from openai import OpenAI
import requests


In [102]:
openai_client = OpenAI(
    api_key=os.getenv("LMSTUDIO_API_KEY"),
    base_url=os.getenv("LMSTUDIO_HOST")
)

In [103]:
# Defining our function to talk the LLM
def llm(prompt):
    response = openai_client.chat.completions.create(
        model="qwen/qwen3.5-9b",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

In [ ]:
# Testing the LLM
llm("What's the answer to life?")

'The question "What is the answer to life?" usually refers to **42**, a famous reference from Douglas Adams\' science fiction comedy series ***The Hitchhiker\'s Guide to the Galaxy***.\n\nIn the story, supercomputer Deep Thought calculates this number after 7.5 million years of processing as the "Answer to the Ultimate Question of Life, the Universe, and Everything." However, the plot reveals that no one actually knows what the *question* itself is. As the character Ford Prefect famously says:\n\n> "The answer to the ultimate question of life, the universe, and everything is 42... But we don\'t know the Ultimate Question of Life, the Universe, and Anything Else."\n\n**Beyond fiction:**\nSince there is no single factual number, the "answer" is largely a matter of personal philosophy. Many people find meaning in:\n*   **Connection**: Love, friendship, and community.\n*   **Purpose**: Creative pursuits, helping others, or mastering a craft.\n*   **Experience**: Cherishing memories, travel

In [ ]:
# Demo how LLM provides an answer with little context in mind
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

Yes, you can usually join a course immediately upon discovering it, but there are a few things to consider depending on **where** the course is hosted:

1.  **If it's an Online Learning Platform (Coursera, edX, Udemy, etc.)**:
    *   Most self-paced courses allow you to enroll instantly. Look for a button that says **"Enroll," "Join," or "Get Started."**
    *   Some paid courses might require checkout, while others may offer an audit option (free listening) with restrictions on grading/certificates.

2.  **If it's an Academic Institution (University/School)**:
    *   Check the **start date** and **enrollment deadline**. Some semester-based courses only open registration during specific windows.
    *   Look for prerequisites or capacity limits.

3.  **Next Steps**:
    *   Click the enrollment button on the course landing page.
    *   Create an account if you don't have one yet.
    *   Complete any necessary profile setup or payment information.

If you tell me the **name of the c

In [107]:
# Adding context
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [108]:
# Building a prompt that includes question and context:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [ ]:
# Updated answer with context; this is Retrival Augmented Generation (RAG)
answer = llm(prompt)
print(answer)

Yes, you can still join the course. However, please note that if you wish to receive a certificate, you must submit your project while submissions are still being accepted.


In [ ]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

Diagram showing how RAG works:

![RAG Diagram](./docs/images/rag_diagram.png)

In [3]:
# Importing external data through a strucuture format for LLM to use
docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [5]:
display(courses_raw)

[{'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 402},
 {'course': 'stock-markets-analytics-zoomcamp',
  'course_name': 'Stock Markets Analytics Zoomcamp',
  'path': '/json/stock-markets-analytics-zoomcamp.json',
  'questions_count': 93},
 {'course': 'ai-dev-tools-zoomcamp',
  'course_name': 'AI Dev Tools Zoomcamp',
  'path': '/json/ai-dev-tools-zoomcamp.json',
  'questions_count': 41},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 79},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 255},
 {'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 472}]

In [6]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1342

Each entry has:

    id - unique identifier
    course - course slug (e.g., machine-learning-zoomcamp)
    section - which section of the course
    question - the FAQ question
    answer - the FAQ answer


In [7]:
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

### Building a Search Function in the RAG Pipeline

In [8]:
# Building the Search Index
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [ ]:
# Demo-ing search with a question
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5}, # assigning weights to specific fields
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [ ]:
# Reviewing all questions in faq search_results
[doc["question"] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'When will the course be offered next?',
 'I missed the first homework - can I still get a certificate?']

In [11]:
# Filtering results by course
results = index.search(
    question,
    num_results=5,
    filter_dict={"course": "mlops-zoomcamp"}
)

In [12]:
results

[{'id': '2d8b16c2a0',
  'course': 'mlops-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course - Can I still join the course after the start date?',
  'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute."},
 {'id': 'c842475338',
  'course': 'mlops-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Homework: Just found this course, can I still submit homeworks?',
  'answer': 'To clarify on **late homework submissions**:\n\n- You cannot submit after the homework is scored, as the form is closed.\n- Once the form is closed (i.e., scored), no further submissions are possible.\n- You can check your code against the solution by reviewing the `homework.md` file.\n\nIf the due date has passed but the form is still "O

In [13]:
# Return all questions listed in search results and related to user initial query
[doc["question"] for doc in results]


['Course - Can I still join the course after the start date?',
 'Homework: Just found this course, can I still submit homeworks?',
 'I forgot if I registered, can I still join the zoomcamp?',
 'Certificate - Can I follow the course in a self-paced mode and get a certificate?',
 'Course: How do I start?']

In [14]:
# Create a executable function
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [15]:
# Now we can call the search function as part of RAG pipeline
search_results = search(question)

In [16]:
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c